In [1]:
from pathlib import Path

PROJECT_ROOT = Path(
    r"D:\Maktab_Sharif_Projects\traffic-vehicle-classification"
)

RAW_ROOT = PROJECT_ROOT / "data" / "raw_without_neysan_in_train"

assert RAW_ROOT.is_dir(), f"Dataset folder not found: {RAW_ROOT}"

print("Dataset location:", RAW_ROOT)
print("Contents:")

for path in sorted(RAW_ROOT.iterdir()):
    print(
        f"  {path.name}",
        "(folder)" if path.is_dir() else "(file)",
    )

Dataset location: D:\Maktab_Sharif_Projects\traffic-vehicle-classification\data\raw_without_neysan_in_train
Contents:
  dataset1 (folder)
  datasetv2_TrainUclean (folder)


# **the next code is about:**


Checking the dataset sources and making sure all class folder names are valid.
It standardizes different names such as nysan and nissan to neysan.
It checks for unexpected class names and stops if any are found.
Finally, it confirms that all five dataset sources passed the checks.

In [2]:
SOURCES = [
    ("dataset1", "train"),
    ("dataset1", "test"),
    ("dataset1", "unclean"),
    ("datasetv2_TrainUclean", "train"),
    ("datasetv2_TrainUclean", "unclean"),
]

TRAINING_CLASSES = {
    "ambulance", "autobus", "kamyun", "kamyunet",
    "minibus", "savari", "taxi", "vanet",
}

LABEL_ALIASES = {
    "nysan": "neysan",
    "nissan": "neysan",
}

for dataset, source in SOURCES:
    source_dir = RAW_ROOT / dataset / source

    assert source_dir.is_dir(), f"Missing source: {source_dir}"

    folders = sorted(
        path.name
        for path in source_dir.iterdir()
        if path.is_dir()
    )

    normalized_labels = {
        LABEL_ALIASES.get(name.strip().lower(), name.strip().lower())
        for name in folders
    }

    unexpected = normalized_labels - (TRAINING_CLASSES | {"neysan"})
    assert not unexpected, (
        f"Unexpected labels in {dataset}/{source}: {unexpected}"
    )

    print(f"{dataset}/{source}: {folders}")

print("\nAll five sources found. Class folder names checked.")

dataset1/train: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'neysan', 'savari', 'taxi', 'vanet']
dataset1/test: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'neysan', 'savari', 'taxi', 'vanet']
dataset1/unclean: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'neysan', 'savari', 'taxi', 'vanet']
datasetv2_TrainUclean/train: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'neysan', 'savari', 'taxi', 'vanet']
datasetv2_TrainUclean/unclean: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus']

All five sources found. Class folder names checked.


# **the next code is about:**

Checking, validating, and creating an inventory of all image files in the datasets.
It verifies that images can be opened, records their size, class labels, and SHA-256 hash.
It also standardizes labels and identifies unreadable or invalid images.
Finally, it creates audit_df with all valid images and their information.

In [3]:
import hashlib
import pandas as pd
from PIL import Image


IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def calculate_sha256(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


rows = []
unreadable = []

for dataset, source in SOURCES:
    source_dir = RAW_ROOT / dataset / source

    for path in sorted(source_dir.rglob("*")):
        if (
            not path.is_file()
            or path.suffix.lower() not in IMAGE_EXTENSIONS
        ):
            continue

        relative_path = path.relative_to(source_dir)

        if len(relative_path.parts) < 2:
            raise ValueError(f"Missing class folder: {path}")

        original_label = relative_path.parts[0].strip().lower()
        label = LABEL_ALIASES.get(original_label, original_label)

        if label not in TRAINING_CLASSES | {"neysan"}:
            raise ValueError(f"Unexpected class: {path}")

        try:
            with Image.open(path) as image:
                image.verify()

            with Image.open(path) as image:
                image.load()
                width, height = image.size

            file_hash = calculate_sha256(path)

        except Exception as exc:
            unreadable.append({
                "path": str(path),
                "error": str(exc),
            })
            continue

        rows.append({
            "path": str(path.resolve()),
            "dataset": dataset,
            "source": source,
            "original_label": original_label,
            "label": label,
            "role": (
                "neysan_test" if label == "neysan"
                else "training_pool"
            ),
            "expected_class": "vanet" if label == "neysan" else label,
            "width": width,
            "height": height,
            "sha256": file_hash,
        })

audit_df = pd.DataFrame(rows)

print("Readable images:", len(audit_df))
print("Unreadable images:", len(unreadable))

if unreadable:
    display(pd.DataFrame(unreadable))
    raise ValueError("Resolve unreadable images before proceeding.")

if audit_df.empty:
    raise ValueError("No images found.")

display(
    audit_df.groupby(["role", "label"])
    .size()
    .rename("images")
    .to_frame()
)

print("Inventory ready; duplicate removal has not been performed yet.")

Readable images: 3536
Unreadable images: 0


images
role          label            
neysan_test   neysan        244
training_pool ambulance     422
              autobus       521
              kamyun        437
              kamyunet      618
              minibus       438
              savari        352
              taxi          348
              vanet         156

Inventory ready; duplicate removal has not been performed yet.


# **the next code is about:**

Finding duplicate images by comparing their SHA-256 hashes.
It checks whether the same image appears in different dataset roles or classes.
It counts duplicate groups, extra copies, and conflicting class labels.
Finally, it displays any problematic duplicates so they can be reviewed.

In [4]:
hash_groups = audit_df.groupby("sha256")

duplicate_rows = audit_df[
    audit_df.duplicated("sha256", keep=False)
].copy()

duplicate_groups = duplicate_rows["sha256"].nunique()
extra_copies = len(audit_df) - audit_df["sha256"].nunique()

role_counts = hash_groups["role"].nunique()
shared_role_hashes = role_counts[role_counts > 1].index

cross_role_rows = audit_df[
    audit_df["sha256"].isin(shared_role_hashes)
].copy()

# Same file assigned different expected classes.
expected_class_counts = hash_groups["expected_class"].nunique()
conflicting_hashes = expected_class_counts[
    expected_class_counts > 1
].index

conflicting_rows = audit_df[
    audit_df["sha256"].isin(conflicting_hashes)
].copy()

print("Duplicate groups:", duplicate_groups)
print("Extra copies:", extra_copies)
print(
    "Groups shared between training pool and Neysan:",
    len(shared_role_hashes),
)
print(
    "Groups with conflicting expected classes:",
    len(conflicting_hashes),
)

if not cross_role_rows.empty:
    print("\nFiles shared between training pool and Neysan:")
    display(
        cross_role_rows[
            ["path", "role", "label", "expected_class", "sha256"]
        ].sort_values(["sha256", "role"])
    )

if not conflicting_rows.empty:
    print("\nConflicting expected classes:")
    display(
        conflicting_rows[
            ["path", "role", "label", "expected_class", "sha256"]
        ].sort_values(["sha256", "label"])
    )

Duplicate groups: 24
Extra copies: 24
Groups shared between training pool and Neysan: 0
Groups with conflicting expected classes: 0


# **the next code is about:**

Removing exact duplicate images and separating the data into the training pool and Neysan test set.
It checks that the 8 training classes and Neysan images are correctly assigned.
It also ensures that no image appears in both groups.
Finally, it saves the cleaned datasets as CSV manifest files for later use.

In [5]:
MANIFEST_DIR = (
    PROJECT_ROOT / "data" / "neysan_experiment" / "manifests"
)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

# Keep one file per exact hash, in a deterministic order.
unique_df = (
    audit_df
    .sort_values(["dataset", "source", "path"])
    .drop_duplicates("sha256", keep="first")
    .reset_index(drop=True)
)

pool_df = unique_df[
    unique_df["role"] == "training_pool"
].copy().reset_index(drop=True)

neysan_df = unique_df[
    unique_df["role"] == "neysan_test"
].copy().reset_index(drop=True)

assert unique_df["sha256"].is_unique
assert set(pool_df["label"]) == TRAINING_CLASSES
assert set(neysan_df["label"]) == {"neysan"}
assert set(neysan_df["expected_class"]) == {"vanet"}
assert set(pool_df["sha256"]).isdisjoint(neysan_df["sha256"])
assert len(pool_df) + len(neysan_df) == len(unique_df)


removed_df = audit_df[
    ~audit_df["path"].isin(unique_df["path"])
].copy()

audit_df.to_csv(MANIFEST_DIR / "all_images.csv", index=False)
pool_df.to_csv(MANIFEST_DIR / "training_pool.csv", index=False)
neysan_df.to_csv(MANIFEST_DIR / "neysan_test.csv", index=False)
removed_df.to_csv(
    MANIFEST_DIR / "excluded_duplicate_copies.csv",
    index=False,
)

print("Original images:", len(audit_df))
print("Unique images:", len(unique_df))
print("Excluded duplicate copies:", len(removed_df))
print("Eight-class pool:", len(pool_df))
print("Held-out Neysan images:", len(neysan_df))

display(
    pool_df.groupby("label").size().rename("images").to_frame()
)

print("Manifests saved:", MANIFEST_DIR)

Original images: 3536
Unique images: 3512
Excluded duplicate copies: 24
Eight-class pool: 3269
Held-out Neysan images: 243


,images
label,
ambulance,408
autobus,520
kamyun,437
kamyunet,614
minibus,434
savari,352
taxi,348
vanet,156


Manifests saved: D:\Maktab_Sharif_Projects\traffic-vehicle-classification\data\neysan_experiment\manifests


# **the next code is about:**
Splitting the 8-class training pool into 80% training and 20% validation data.
It keeps the classes balanced using stratify and uses SEED = 42 for reproducibility.
It checks that there are no duplicate images shared between the splits or Neysan test set.
Finally, it saves the training, validation, and class-count information as CSV files.

In [6]:
from sklearn.model_selection import train_test_split

SEED = 42

train_df, val_df = train_test_split(
    pool_df,
    test_size=0.20,
    stratify=pool_df["label"],
    random_state=SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

assert set(train_df["label"]) == TRAINING_CLASSES
assert set(val_df["label"]) == TRAINING_CLASSES

assert set(train_df["sha256"]).isdisjoint(val_df["sha256"])
assert set(train_df["sha256"]).isdisjoint(neysan_df["sha256"])
assert set(val_df["sha256"]).isdisjoint(neysan_df["sha256"])

assert len(train_df) + len(val_df) == len(pool_df)

split_counts = pd.DataFrame({
    "train": train_df["label"].value_counts(),
    "validation": val_df["label"].value_counts(),
}).fillna(0).astype(int).sort_index()

split_counts["total"] = (
    split_counts["train"] + split_counts["validation"]
)

train_df.to_csv(MANIFEST_DIR / "train.csv", index=False)
val_df.to_csv(MANIFEST_DIR / "validation.csv", index=False)
split_counts.to_csv(MANIFEST_DIR / "split_class_counts.csv")

display(split_counts)

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Held-out Neysan images:", len(neysan_df))
print("Exact hashes shared between splits: 0")
print("Split manifests saved:", MANIFEST_DIR)

,train,validation,total
label,,,
ambulance,326,82,408
autobus,416,104,520
kamyun,350,87,437
kamyunet,491,123,614
minibus,347,87,434
savari,282,70,352
taxi,278,70,348
vanet,125,31,156


Training images: 2615
Validation images: 654
Held-out Neysan images: 243
Exact hashes shared between splits: 0
Split manifests saved: D:\Maktab_Sharif_Projects\traffic-vehicle-classification\data\neysan_experiment\manifests
